# Feature Engineering

**Plan 2 - Tasks 3 & 4**

Encode categorical variables and create behavioral features from the cleaned dataset.

## Step 1: Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv('../datasets/student_exam_cleaned.csv')
print(f"Loaded: {df.shape}")
df.head()

Loaded: (10000, 18)


,student_id,gender,age,parental_education,family_income,internet_access,study_environment,study_hours_per_day,attendance_rate,sleep_hours,social_media_hours,assignment_completion_rate,online_courses_completed,tutoring,final_exam_score,previous_gpa,pass_fail,grade_category
0,S00001,Male,17,High School,Medium,Yes,Quiet,2.98,96.5,6.05,0.1,80.5,1,Yes,49.1,2.44,Fail,F
1,S00002,Female,18,High School,Low,Yes,Quiet,4.45,95.7,6.96,2.9,70.9,0,Yes,70.1,2.79,Pass,C
2,S00003,Male,17,High School,Medium,No,Quiet,3.75,76.0,7.02,2.4,77.6,4,Yes,42.2,1.49,Fail,F
3,S00004,Male,18,Bachelor,Medium,Yes,Quiet,2.03,72.6,6.23,3.5,63.5,4,No,31.9,1.34,Fail,F
4,S00005,Male,18,Bachelor,Medium,Yes,Quiet,5.14,87.3,8.54,2.1,71.8,0,No,66.4,2.60,Pass,C


## Step 2: Binary Encode Yes/No and Male/Female

In [2]:
binary_map = {'Yes': 1, 'No': 0}
df['internet_access_enc'] = df['internet_access'].map(binary_map)
df['tutoring_enc'] = df['tutoring'].map(binary_map)
df['gender_enc'] = df['gender'].map({'Male': 0, 'Female': 1})
print("Binary encoding: internet_access_enc, tutoring_enc, gender_enc")

Binary encoding: internet_access_enc, tutoring_enc, gender_enc


## Step 3: Ordinal Encode Ordered Categories

In [3]:
df['parental_education_enc'] = df['parental_education'].map({
    'High School': 0, 'Bachelor': 1, 'Master': 2, 'Phd': 3
})
df['family_income_enc'] = df['family_income'].map({
    'Low': 0, 'Medium': 1, 'High': 2
})
df['study_environment_enc'] = df['study_environment'].map({
    'Noisy': 0, 'Moderate': 1, 'Quiet': 2
})
df['pass_fail_enc'] = df['pass_fail'].map({'Fail': 0, 'Pass': 1})
print("Ordinal encoding: parental_education_enc, family_income_enc, study_environment_enc, pass_fail_enc")

Ordinal encoding: parental_education_enc, family_income_enc, study_environment_enc, pass_fail_enc


## Step 4: One-Hot Encode grade_category

In [4]:
grade_dummies = pd.get_dummies(df['grade_category'], prefix='grade', dtype=int)
df = pd.concat([df, grade_dummies], axis=1)
print(f"One-hot encoded grade_category: {list(grade_dummies.columns)}")

One-hot encoded grade_category: ['grade_A', 'grade_B', 'grade_C', 'grade_D', 'grade_F']


## Step 5: Create Engineered Behavioral Features

In [5]:
# Total productive hours (study + sleep)
df['productive_hours'] = df['study_hours_per_day'] + df['sleep_hours']

# NOTE: 'screen_time_ratio' (social_media_hours / 16) was REMOVED.
# A constant denominator makes it a pure linear rescale of social_media_hours
# (identical correlation, no effect on tree models or StandardScaler-based LR),
# so it carried no information beyond social_media_hours itself.

# Study x Attendance interaction
# Note: study_hours (0-10) * attendance_rate (0-100) produces scale 0-1000.
# This does NOT affect results because:
# - Correlations are invariant to constant scaling
# - RF is scale-invariant; LR uses StandardScaler which normalizes
df['study_attendance'] = df['study_hours_per_day'] * df['attendance_rate']

# Engagement score: behavioral engagement (no internal scores)
df['engagement_score'] = (df['assignment_completion_rate'] + df['attendance_rate']) / 2

# Disadvantage index (no internet + no tutoring + low income)
df['disadvantage_index'] = (
    (1 - df['internet_access_enc']) +
    (1 - df['tutoring_enc']) +
    (df['family_income_enc'] == 0).astype(int)
)

# Social media to study ratio (distraction vs effort)
# Epsilon guard prevents division by zero if study_hours_per_day = 0
EPSILON = 1e-8
df['social_study_ratio'] = df['social_media_hours'] / (df['study_hours_per_day'] + EPSILON)

print("Created 5 behavioral features (screen_time_ratio removed as redundant)")

Created 5 behavioral features (screen_time_ratio removed as redundant)


## Step 6: Verify New Features

In [6]:
new_features = ['productive_hours', 'study_attendance',
                'engagement_score', 'disadvantage_index', 'social_study_ratio']
for feat in new_features:
    nan_count = df[feat].isna().sum()
    inf_count = np.isinf(df[feat]).sum()
    print(f"{feat}: NaN={nan_count}, Inf={inf_count}, Range=[{df[feat].min():.2f}, {df[feat].max():.2f}]")

# Assert no Inf from division
assert df['social_study_ratio'].isin([np.inf, -np.inf]).sum() == 0, "Division by zero in social_study_ratio"
print("\nAll assertions passed: no Inf values in engineered features")

productive_hours: NaN=0, Inf=0, Range=[4.67, 15.82]
study_attendance: NaN=0, Inf=0, Range=[29.90, 657.45]
engagement_score: NaN=0, Inf=0, Range=[51.10, 100.00]
disadvantage_index: NaN=0, Inf=0, Range=[0.00, 3.00]
social_study_ratio: NaN=0, Inf=0, Range=[0.00, 14.00]

All assertions passed: no Inf values in engineered features


## Step 7: Save Feature-Engineered Dataset

In [7]:
df.to_csv('../datasets/student_exam_features.csv', index=False)
print(f"Feature-engineered dataset saved: {df.shape}")
print(f"Columns: {list(df.columns)}")

Feature-engineered dataset saved: (10000, 35)
Columns: ['student_id', 'gender', 'age', 'parental_education', 'family_income', 'internet_access', 'study_environment', 'study_hours_per_day', 'attendance_rate', 'sleep_hours', 'social_media_hours', 'assignment_completion_rate', 'online_courses_completed', 'tutoring', 'final_exam_score', 'previous_gpa', 'pass_fail', 'grade_category', 'internet_access_enc', 'tutoring_enc', 'gender_enc', 'parental_education_enc', 'family_income_enc', 'study_environment_enc', 'pass_fail_enc', 'grade_A', 'grade_B', 'grade_C', 'grade_D', 'grade_F', 'productive_hours', 'study_attendance', 'engagement_score', 'disadvantage_index', 'social_study_ratio']
